## jiyuan yang

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import silhouette_score
from scipy.spatial.distance import squareform
# 假设 scembedding 是 (n_cells, n_features) 的 numpy array
# y 是长度为 n_cells 的标签数组（cluster labels）
# 方法 1：使用 scipy 的 pdist with 'correlation' (推荐)
from scipy.spatial.distance import pdist, squareform
gt=pd.read_csv('/home/qyyuan/project/ST_CP/JE/compare/groundtruth/NM_yangjy_X/data/S3_GT.txt',sep='\t',header=0,index_col=0)

In [2]:
import scanpy as sc
celltype_label=pd.read_csv('/home/qyyuan/project/ST_CP/JE/method/data/scRNA_subsampled_50k_cleaned_metadata.csv',sep=',',index_col=0)

/home/qyyuan/anaconda3/envs/GPU/lib/python3.11/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [3]:
adata_sc=sc.read('/home/qyyuan/project/ST_CP/JE/method/data/scRNA_subsampled_50k_cleaned.h5ad')
adata_st=sc.read("/home/qyyuan/project/ST_CP/JE/compare/groundtruth/NM_yangjy_X/data/Visium_FAD.h5ad")

In [4]:
import numpy as np
from scipy.stats import pearsonr
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import silhouette_score
from scipy.spatial.distance import squareform
# 假设 scembedding 是 (n_cells, n_features) 的 numpy array
# y 是长度为 n_cells 的标签数组（cluster labels）
# 方法 1：使用 scipy 的 pdist with 'correlation' (推荐)
from scipy.spatial.distance import pdist, squareform
def evaluate_predictions_numpy(
    X_sc_true,              # same shape as predictsc (optional, can be None)
    X_st_true,         # (n_spots, n_genes)
    predictsc=None,              # (n_sc_cells, n_genes) or embedding
    scembeding=None,
    predictst=None,         # (n_spots, n_genes)
    predictA=None,          # (n_spots, n_sc_cells)
    A_gt=None,              # (n_spots, n_sc_cells) or spot × cell assignment
    cell_type_labels=None, 
    A_deconv=None, # list[str], length = n_sc_cells
    A_gt_deconv=None,       # (n_spots, n_types)
    cell_types_gt_order=None
):
    """
    Pure numpy evaluation.
    Some inputs can be None.
    """

    metrics = {}

    # Ensure numpy
    if (predictsc is not None) and (X_sc_true is not None):
        predictsc = np.asarray(predictsc, dtype=np.float32)
        X_sc_true = np.asarray(X_sc_true, dtype=np.float32)
    if (predictst is not None) and (X_st_true is not None):
        predictst = np.asarray(predictst, dtype=np.float32)
        X_st_true = np.asarray(X_st_true, dtype=np.float32)
    if predictA is not None:
        predictA = np.asarray(predictA, dtype=np.float32)
    if A_gt is not None:
        A_gt = np.asarray(A_gt, dtype=np.float32)

    # -------------------------------
    # 1️⃣ Expression PCC
    # -------------------------------
    def mean_pcc(x_true, x_pred):
        row_pccs, col_pccs = [], []
        for i in range(x_true.shape[0]):
            t, p = x_true[i], x_pred[i]
            if np.std(t) > 1e-8 and np.std(p) > 1e-8:
                r, _ = pearsonr(t, p)
                row_pccs.append(float(r))
            else:
                row_pccs.append(0.0)
        for j in range(x_true.shape[1]):
            t, p = x_true[:, j], x_pred[:, j]
            if np.std(t) > 1e-8 and np.std(p) > 1e-8:
                r, _ = pearsonr(t, p)
                col_pccs.append(float(r))
            else:
                col_pccs.append(0.0)
        return np.mean(row_pccs), np.mean(col_pccs)

    if predictsc is not None and X_sc_true is not None:
        f_row_pcc, f_col_pcc = mean_pcc(X_sc_true, predictsc)
        metrics['predictsc_row_pcc'] = float(f_row_pcc)
        metrics['predictsc_col_pcc'] = float(f_col_pcc)

    if predictst is not None and X_st_true is not None:
        st_row_pcc, st_col_pcc = mean_pcc(X_st_true, predictst)
        metrics['predictst_row_pcc'] = float(st_row_pcc)
        metrics['predictst_col_pcc'] = float(st_col_pcc)

    # -------------------------------
    # 2️⃣ Assignment Accuracy
    # -------------------------------
    if predictA is not None and A_gt is not None:
        pred_assign = np.argmax(predictA, axis=1)
        true_assign = np.argmax(A_gt, axis=1)
        metrics['assignment_accuracy'] = float(np.mean(pred_assign == true_assign))

    # -------------------------------
    # 3️⃣ Deconvolution
    # -------------------------------
    if A_gt_deconv is not None:
        A_gt_deconv = np.asarray(A_gt_deconv, dtype=np.float32)
    if A_deconv is not None:
        dec_pred_aligned=A_deconv.values
    else:
        dec_pred_aligned = None
        if predictA is not None and cell_type_labels is not None:
            le = LabelEncoder()
            labels = le.fit_transform(cell_type_labels)
            n_types = len(le.classes_)
            one_hot_sc = np.eye(n_types)[labels]
            dec_pred_raw = predictA @ one_hot_sc  # (n_spots, n_types)

            if cell_types_gt_order is not None:
                #A_gt_deconv = np.asarray(A_gt_deconv, dtype=np.float32)
                pred_order = le.classes_.tolist()
                gt_order = cell_types_gt_order
                reorder_idx = [pred_order.index(ct) for ct in gt_order]
                dec_pred_aligned = dec_pred_raw[:, reorder_idx]
            else:
                print('Please provide cell_types_gt_order')
            # Row-wise PCC
    if (dec_pred_aligned is not None) and (A_gt_deconv is not None):
        pcc_list = []
        for i in range(A_gt_deconv.shape[0]):
            gt_i = A_gt_deconv[i]
            pred_i = dec_pred_aligned[i]
            if np.std(gt_i) < 1e-8 or np.std(pred_i) < 1e-8:
                r = 0.0
            else:
                r, _ = pearsonr(gt_i, pred_i)
                r = float(r) if np.isfinite(r) else 0.0
            pcc_list.append(r)
            metrics['deconv_mean_pcc'] = float(np.mean(pcc_list))
            metrics['deconv_mean_mae'] = float(np.mean(np.abs(A_gt_deconv - dec_pred_aligned)))
            metrics['deconv_mean_rmse'] = float(np.sqrt(np.mean((A_gt_deconv - dec_pred_aligned) ** 2)))
            metrics['PCC_all']=pcc_list

    # -------------------------------
    # 4️⃣ Silhouette Index (NEW)
    # -------------------------------
    if scembeding is not None and cell_type_labels is not None:
        le = LabelEncoder()
        y = le.fit_transform(cell_type_labels)
        if len(np.unique(y)) > 1:
            correlation_distances = pdist(scembeding, metric='correlation')  # shape: (n*(n-1)/2,)
            distance_matrix = squareform(correlation_distances)  # shape: (n, n)
            sil = silhouette_score(distance_matrix, y, metric='precomputed')
            # sil= silhouette_score(scembeding, y, metric='euclidean')
            metrics['silhouette_index'] = float(sil)
        else:
            metrics['silhouette_index'] = 0.0

    print("✅ Evaluation complete.")
    # -------------------------------
    # 5️⃣ Visualization: per-cell-type gene scatter (NEW)
    # -------------------------------
    if predictsc is not None and X_sc_true is not None and cell_type_labels is not None:
        import matplotlib.pyplot as plt

        predictsc = np.asarray(predictsc)
        X_sc_true = np.asarray(X_sc_true)

        cell_type_labels = np.asarray(cell_type_labels)
        unique_types = np.unique(cell_type_labels)

        # 固定最多画 10 个 cell type
        unique_types = unique_types[:10]
        n_types = len(unique_types)

        n_cols = 5
        n_rows = int(np.ceil(n_types / n_cols))

        fig, axes = plt.subplots(
            n_rows, n_cols,
            figsize=(4 * n_cols, 4 * n_rows),
            squeeze=False
        )

        for idx, ct in enumerate(unique_types):
            row = idx // n_cols
            col = idx % n_cols
            ax = axes[row, col]

            # 找到该 cell type 的所有 cell
            ct_indices = np.where(cell_type_labels == ct)[0]
            if len(ct_indices) == 0:
                ax.axis("off")
                continue

            # 抽一个 cell（固定第一个，保证可复现；也可以改成 np.random.choice）
            cell_idx = ct_indices[0]

            x = X_sc_true[cell_idx]
            y = predictsc[cell_idx]

            # 计算 PCC
            if np.std(x) > 1e-8 and np.std(y) > 1e-8:
                r, _ = pearsonr(x, y)
            else:
                r = 0.0

            ax.scatter(x, y, s=6, alpha=0.6)
            ax.set_title(f"{ct}\nPCC={r:.2f}", fontsize=10)
            ax.set_xlabel("X_sc_true")
            ax.set_ylabel("predictsc")

        # 关掉多余子图
        for j in range(idx + 1, n_rows * n_cols):
            axes[j // n_cols, j % n_cols].axis("off")

        plt.tight_layout()
        plt.show()
    return metrics

### novosparc

In [5]:
A_pred=np.load("/home/qyyuan/project/ST_CP/JE/compare/novosparc/A_alpha_1_jiyanyang-alpha1-epsilon-3.npy")

In [6]:
A_pred_df=pd.DataFrame(A_pred,index=adata_sc.obs_names,columns=adata_st.obs_names)
A_pred_df=A_pred_df[gt.index]

In [7]:
result_novo=evaluate_predictions_numpy(
    adata_sc.X,              # same shape as predictsc (optional, can be None)
    adata_st.X,         # (n_spots, n_genes)
    predictsc=None,              # (n_sc_cells, n_genes) or embedding
    scembeding=None,
    predictst=None,         # (n_spots, n_genes)
    predictA=A_pred_df.T,          # (n_spots, n_sc_cells)
    A_gt=None,              # (n_spots, n_sc_cells) or spot × cell assignment
    cell_type_labels=celltype_label['cell_type'].values,  # list[str], length = n_sc_cells
    A_gt_deconv=gt,       # (n_spots, n_types)
    cell_types_gt_order=gt.columns
)

✅ Evaluation complete.


In [8]:
result_novo

{'deconv_mean_pcc': 0.41877841949762495,
 'deconv_mean_mae': 0.03703811794575486,
 'deconv_mean_rmse': 0.11524359816484975,
 'PCC_all': [0.1343328626675684,
  0.5369811957291546,
  0.09355273707470281,
  0.26409869730494023,
  -0.027634715680672287,
  0.5616014433511753,
  0.29637342344063583,
  0.3496958568046201,
  0.003563054618497162,
  0.5416985787358579,
  0.1108326408139442,
  0.45045078952787,
  0.40425560122218585,
  0.5850419728846752,
  0.38994443542062496,
  0.808325248062288,
  0.3629771494543941,
  0.3292167741565923,
  0.09061347169915522,
  0.3546371573769866,
  0.6236908864760305,
  0.5924329111214531,
  0.29760546387272513,
  0.5931993395817106,
  -0.02926513367035554,
  0.42826111289968144,
  0.6981908636508959,
  0.1023347287101917,
  0.11737866715928508,
  0.6148481566077917,
  0.416541597456598,
  0.05701700865782516,
  0.05493099932963128,
  0.3243847920575321,
  0.49858663057520064,
  0.24757139248768495,
  0.5642007351131474,
  0.4669710342681982,
  0.490021818

### tangram

In [21]:
import scanpy as sc
pred_st=sc.read('/home/qyyuan/project/ST_CP/JE/Tangram/spatial_with_tangram-Version3-work.h5ad')
pred_st_df=pd.DataFrame(pred_st.X.toarray())
pred_st_df.index=pred_st.obs_names
pred_st_df.columns=pred_st.var_names
pred_A = sc.read('/home/qyyuan/project/ST_CP/JE/Tangram/tangram_mapping-Version3-work.h5ad')


In [5]:
labels=celltype_label['cell_type'].values

In [6]:
count_file_path='/home/qyyuan/project/ST_CP/JE/method/stardist/count_spot/spot_loc_with_counts_r_f.csv'
Count_cell=pd.read_csv(count_file_path)

In [27]:
max_indices = np.argmax(pred_A.X, axis=1)

In [35]:
max_values = np.max(pred_A.X, axis=1)

In [36]:
max_values

array([0.25662705, 0.993196  , 0.3051832 , ..., 0.50746125, 0.57759446,
       0.9964347 ], shape=(49985,), dtype=float32)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.collections import PatchCollection
from PIL import Image

# ========= 1. 读取图像 =========
img_path = "/home/qyyuan/project/ST_CP/JE/compare/groundtruth/NM_yangjy_X/data/VisiumImage_FAD_1.jpg"
img = np.array(Image.open(img_path))

H, W = img.shape[:2]
print("Image shape:", img.shape)
#spot_loc=pd.read_csv('/home/qyyuan/project/ST_CP/JE/method/stardist/count_spot/spot_loc_with_counts_r_f.csv',sep=',',index_col=0)

/home/qyyuan/.conda/envs/GPU/lib/python3.10/site-packages/PIL/Image.py:3451: DecompressionBombWarning: Image size (99010000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Image shape: (10000, 9901, 3)


In [32]:
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.patches import Circle
import numpy as np
import random
import os

x = Count_cell['imagecol'].values[max_indices]
y = Count_cell['imagerow'].values[max_indices]
# 参数设置
r_base = 73.1 * 100 / 118 / 2  # 原始半径计算
labels = np.array(labels)      # 转为 numpy 数组方便筛选
unique_labels = sorted(list(set(labels)))

print(f"检测到 {len(unique_labels)} 种细胞类型，将生成 {len(unique_labels)} 张独立图片。")

# 创建输出目录 (可选，防止文件散落)
output_dir = "/home/qyyuan/project/ST_CP/JE/A_matrix/count_spot/single_type_plots"
os.makedirs(output_dir, exist_ok=True)

# ==========================================
# 主循环：为每一种 Label 单独画图
# ==========================================
for i, current_label in enumerate(unique_labels):
    # 1. 筛选当前类型的坐标
    mask = (labels == current_label)
    curr_x = x[mask]
    curr_y = y[mask]
    
    # 如果该类型没有点，跳过
    if len(curr_x) == 0:
        continue
        
    # 2. 计算随机偏移 (逻辑同原代码，但只针对当前类型的点)
    new_x_coords = []
    new_y_coords = []
    
    # 确定当前这批点的半径
    if np.isscalar(r_base):
        radii = np.full(len(curr_x), r_base)
    else:
        # 如果 r 是数组，也需要对应筛选
        # 假设原始的 r 数组叫 r_array，这里需要类似 mask 的筛选
        # 此处假设 r 是标量，如果是数组请自行添加筛选逻辑: radii = np.array(r)[mask]
        radii = np.full(len(curr_x), r_base)

    new_r = radii / 3  # 缩小半径
    
    for j in range(len(curr_x)):
        cx, cy = curr_x[j], curr_y[j]
        radius = radii[j]
        
        # 极坐标采样生成随机偏移
        theta = random.uniform(0, 2 * np.pi)
        dist = np.sqrt(random.uniform(0, 1)) * radius 
        
        nx = cx + dist * np.cos(theta)
        ny = cy + dist * np.sin(theta)
        
        new_x_coords.append(nx)
        new_y_coords.append(ny)

    # 3. 开始绘图
    fig, ax = plt.subplots(figsize=(12, 12))
    
    # 显示背景图 (所有图背景相同)
    ax.imshow(img, interpolation='bilinear')
    
    # 创建黑色圆点集合
    patches = []
    # 所有点的颜色都是黑色
    face_colors = ['black'] * len(new_x_coords)
    
    for xi, yi, r_val in zip(new_x_coords, new_y_coords, new_r):
        display_r = max(r_val, 2.0) 
        patches.append(Circle((xi, yi), display_r))

    p = PatchCollection(patches,
                        facecolor=face_colors,
                        edgecolor=None,
                        linewidth=0,          # 黑色实心点通常不需要边框，或者设为黑色
                        rasterized=False)
    ax.add_collection(p)
    
    # 设置坐标轴
    ax.set_aspect('equal')
    ax.axis('off')
    # 标题包含细胞类型名称，方便识别
    safe_filename = str(current_label).replace('/', '_').replace('\\', '_') # 防止文件名含非法字符
    ax.set_title(f"Cell Type: {current_label}", fontsize=16, pad=20, color='black')
    
    # 4. 保存文件
    # 文件名格式：CellType_TypeA.pdf
    output_path = os.path.join(output_dir, f"CellType_{safe_filename}.pdf")
    
    plt.savefig(
        output_path, 
        format='pdf', 
        bbox_inches='tight', 
        dpi=600,
        metadata={'Creator': f'Single Type Plot: {current_label}'}
    )
    
    # 关闭当前图形以释放内存，避免画27张图后内存溢出
    plt.close(fig)
    
    # 打印进度
    if (i + 1) % 5 == 0 or i == len(unique_labels) - 1:
        print(f"进度：{i+1}/{len(unique_labels)} - 已保存：{output_path}")

print(f"\n完成！所有 {len(unique_labels)} 张单细胞类型图已保存至：{output_dir}")

检测到 27 种细胞类型，将生成 27 张独立图片。
进度：5/27 - 已保存：/home/qyyuan/project/ST_CP/JE/A_matrix/count_spot/single_type_plots/CellType_Endo.pdf
进度：10/27 - 已保存：/home/qyyuan/project/ST_CP/JE/A_matrix/count_spot/single_type_plots/CellType_L5 PT CTX.pdf
进度：15/27 - 已保存：/home/qyyuan/project/ST_CP/JE/A_matrix/count_spot/single_type_plots/CellType_L6b CTX.pdf
进度：20/27 - 已保存：/home/qyyuan/project/ST_CP/JE/A_matrix/count_spot/single_type_plots/CellType_Oligo.pdf
进度：25/27 - 已保存：/home/qyyuan/project/ST_CP/JE/A_matrix/count_spot/single_type_plots/CellType_Sst Chodl.pdf
进度：27/27 - 已保存：/home/qyyuan/project/ST_CP/JE/A_matrix/count_spot/single_type_plots/CellType_Vip.pdf

完成！所有 27 张单细胞类型图已保存至：/home/qyyuan/project/ST_CP/JE/A_matrix/count_spot/single_type_plots


In [15]:
type(pred_A.X)

numpy.ndarray

In [20]:
pred_A.X.max()

np.float32(0.999146)

In [23]:
len(set(max_indices))

3464

In [7]:
gt_st=pd.DataFrame(adata_st.X.toarray(),index=adata_st.obs_names,columns=adata_st.var_names)

NameError: name 'adata_st' is not defined

In [11]:
cgene=list(set(gt_st.columns.str.lower()) & set(pred_st_df.columns.str.lower()))
gt_st.columns=gt_st.columns.str.lower()
pred_st_df.columns=pred_st_df.columns.str.lower()

In [12]:
gt_st=gt_st[cgene]
pred_st_df=pred_st_df[cgene]

In [ ]:
A_pred_df=pd.DataFrame(pred_A.X,index=adata_sc.obs_names,columns=adata_st.obs_names)
A_pred_df=A_pred_df[gt.index]

NameError: name 'adata_sc' is not defined

In [14]:
A_dev=pred_st.obsm['tangram_ct_pred']
A_dev=A_dev.loc[gt.index]
A_dev=A_dev[gt.columns]


In [15]:
result_tangram=evaluate_predictions_numpy(
    adata_sc.X,              # same shape as predictsc (optional, can be None)
    gt_st.values,         # (n_spots, n_genes)
    predictsc=None,              # (n_sc_cells, n_genes) or embedding
    scembeding=None,
    predictst=None,         # (n_spots, n_genes)
    predictA=None,          # (n_spots, n_sc_cells)
    A_gt=None,              # (n_spots, n_sc_cells) or spot × cell assignment
    cell_type_labels=celltype_label['cell_type'].values,  # list[str], length = n_sc_cells
    A_deconv=A_dev,
    A_gt_deconv=gt,       # (n_spots, n_types)
    cell_types_gt_order=gt.columns
)

✅ Evaluation complete.


### RCTD

In [17]:
pred_A=pd.read_csv("/home/qyyuan/project/ST_CP/JE/compare/RCTD/deconv_jiyuanyang.txt",sep='\t',header=0,index_col=0)
# 将列名中的 '/' 替换为 ' ' (空格)
gtcol = gt.columns.str.replace('/', ' ', regex=False)
pred_A=pred_A.loc[gt.index]
pred_A=pred_A[gtcol]

In [18]:
result_1=evaluate_predictions_numpy(
    adata_sc.X,              # same shape as predictsc (optional, can be None)
    adata_st.X,         # (n_spots, n_genes)
    predictsc=None,              # (n_sc_cells, n_genes) or embedding
    scembeding=None,
    predictst=None,         # (n_spots, n_genes)
    predictA=None,          # (n_spots, n_sc_cells)
    A_gt=None,              # (n_spots, n_sc_cells) or spot × cell assignment
    cell_type_labels=celltype_label['cell_type'].values,  # list[str], length = n_sc_cells
    A_deconv=pred_A,
    A_gt_deconv=gt,       # (n_spots, n_types)
    cell_types_gt_order=gt.columns
)

✅ Evaluation complete.


### spotify

In [7]:
A_dev_pred=pd.read_csv('/home/qyyuan/project/ST_CP/JE/compare/spotiphy/Spotiphy_Result/proportion.csv',index_col=0)
A_dev_pred=A_dev_pred.loc[gt.index]
A_dev_pred=A_dev_pred[gt.columns]

In [8]:
result_s=evaluate_predictions_numpy(
    adata_sc.X,              # same shape as predictsc (optional, can be None)
    adata_st.X,         # (n_spots, n_genes)
    predictsc=None,              # (n_sc_cells, n_genes) or embedding
    scembeding=None,
    predictst=None,         # (n_spots, n_genes)
    predictA=None,          # (n_spots, n_sc_cells)
    A_gt=None,              # (n_spots, n_sc_cells) or spot × cell assignment
    cell_type_labels=celltype_label['cell_type'].values,  # list[str], length = n_sc_cells
    A_deconv=A_dev_pred,
    A_gt_deconv=gt,       # (n_spots, n_types)
    cell_types_gt_order=gt.columns
)

✅ Evaluation complete.


In [10]:
result_s

{'deconv_mean_pcc': 0.5498770450222634,
 'deconv_mean_mae': 0.03921586876894069,
 'deconv_mean_rmse': 0.09232249952354311,
 'PCC_all': [0.5801236563145744,
  0.6934359592998225,
  0.42211331094712584,
  0.5653288355149012,
  0.6419607076852337,
  0.5028812669916719,
  0.7340721635187379,
  0.31449207606544094,
  0.7388922660199703,
  0.8306190366382854,
  0.9047330665741604,
  0.5885381945069699,
  0.4557384058579817,
  0.8029325396542004,
  0.552833741538648,
  0.7294741359162624,
  0.05751019634566877,
  0.2285146810730113,
  0.6691860320554547,
  0.058519556047472394,
  0.37919173142010804,
  0.9781011075561873,
  0.30467314973094617,
  0.8045006213195782,
  0.9832758520908859,
  0.520712600501007,
  0.5906391702725238,
  0.06248581976730315,
  0.006477579892212615,
  0.8600652707662692,
  0.70494029155441,
  -0.01704010618138581,
  0.948873951918259,
  0.6498791666469121,
  0.805418201925698,
  0.5693991388574272,
  0.8951785165539141,
  0.6152960829709463,
  0.790208079434355,
  0

In [ ]:
result_all=pd.DataFrame(result_s['PCC_all'],index=A_dev_pred.index,columns=['Spotiphy'])
result_all['Novosaprc']=result_novo['PCC_all']

In [26]:
result_all['Trangram']=result_tangram['PCC_all']

In [28]:
result_all['RCTD']=result_1['PCC_all']

In [29]:
result_all.to_csv('temp_jiyuanyang_compare_docov.txt',sep='\t')